# PAPER-EXP-08B
## Once-Only Zero-Shot DERM12345 Evaluation of Locked PAPER-EXP-08A

Run unchanged on Kaggle with T4 GPU and Internet OFF. The model is already locked; this notebook performs no training or selection.


In [1]:
from __future__ import annotations

import contextlib
import hashlib
import io
import json
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, recall_score, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# =============================================================================
# PAPER-EXP-08B: ONCE-ONLY ZERO-SHOT DERM12345 EVALUATION OF LOCKED 08A
# =============================================================================

EXPERIMENT = "PAPER-EXP-08B"
TITLE = "Once-Only Zero-Shot DERM12345 Evaluation of Locked PAPER-EXP-08A"
CODE_VERSION = "PAPEREXP08B_LOCKED_08A_DERM12345_ZEROSHOT_2026-08-30_v1"
LOCK_ID = "PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK_2026-08-30_v1"

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
OUT = WORKING_ROOT / 'PAPEREXP08B_Locked_08A_DERM12345_ZeroShot'
AUDIT = OUT / 'audit'
TABLES = OUT / 'tables'
PRED = OUT / 'predictions'
for p in (OUT, AUDIT, TABLES, PRED):
    p.mkdir(parents=True, exist_ok=True)
TMP = WORKING_ROOT / 'paperexp08b_verified_components'
TMP.mkdir(exist_ok=True)

CLASSES = ('MEL', 'NV', 'BCC', 'AKIEC', 'BKL')
C2I = {c:i for i,c in enumerate(CLASSES)}
DOMAINS = ('ISIC2018_HAM10000', 'BCN20000', 'Derm7pt', 'PAD_UFES_20')

IMAGE_SIZE = 300
FEATURE_BATCH_B0 = 32
FEATURE_BATCH_B3 = 16
WORKERS = 4
RESIDUAL_LOGIT_CAP = 0.50
DROPOUT = 0.10
ROUTER_TEMPERATURE = 1.0
GLOBAL_RESIDUAL_GAIN = 1.05
EXTS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
DERM_PATH_TOKENS = ('derm12345', 'derm-12345', 'derm_12345')

EXPECTED_08A_RECOVERY_SHA = 'f6620c2b70febd921af9622b3a9ab8e7f9e4c5488bff5b1ea07986761c08a7d9'
EXPECTED_08A_LOCK_NAME = 'PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK.json'
EXPECTED_08A_LOCK_SHA = '831b4d52e835bd2052d8bff20ec33506100bbd519a60b0f75ffe94446b3c2e84'
DERM_MANIFEST_NAME = 'litegen06a_derm12345_domain_holdout.csv'
DERM_MANIFEST_SHA = 'f42a2eb0ccd3ef34613dcb5e0d93b60071300af134794409cfce4dab50c7ffe5'
DERM_ROWS = 2321
DERM_GROUPS = 574
DERM_CLASS_COUNTS = {'MEL':82, 'NV':2000, 'BCC':85, 'AKIEC':19, 'BKL':135}

BASELINE_CKPT_HASHES = {
    'b0_seed_20260820':'51eb6d967c6805ad622863c9b2d586954a119c2330a6608e9a73c95b808e6932',
    'b0_seed_20260821':'a32e5e6679af64db64d7154d59dcf62d7f8ef230b3f029d0c610168ea49998c3',
    'b0_seed_20260822':'610a79ffda7719598d6b26009dfe1c5534a4c7951179f9b018a5381e04bf7f6d',
    'b3_seed_20260820':'a5d882e39255417a7171a5403b4298430d8ac0c3af7e31dd589538c73e950b19',
    'b3_seed_20260821':'8827e01a59205723c5fefad9449740129072edb07bd4870c1361d904e512bbdb',
    'b3_seed_20260822':'33b60778f13785ca7748d85a32421bb53a9044f23ed716aafb1c625238463e98',
}
ADAPTER_HASHES = {
    'adapter_seed_20260829':'d96365b453a9306955f0025930629710d49e30a1eaf1950ca1784371afd6c667',
    'adapter_seed_20260830':'93ee73ffe2b6b4f65aa2ba61881b18646cf8a874324d3a60018e89c0ac77014d',
    'adapter_seed_20260831':'03b443c8e0b9bf9dd886397fbc45ae49ad0b18556919edb58f90a17ca0949e83',
}
B0_NAMES = tuple(k for k in BASELINE_CKPT_HASHES if k.startswith('b0_'))
B3_NAMES = tuple(k for k in BASELINE_CKPT_HASHES if k.startswith('b3_'))
SELECTED_ADAPTER_EPOCH = 3


def sha_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def json_dump(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2), encoding='utf-8')

@contextlib.contextmanager
def amp(device: torch.device):
    if device.type == 'cuda':
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            yield
    else:
        yield


def exact_bytes(name: str, expected_sha: str) -> tuple[bytes, str]:
    for p in INPUT_ROOT.rglob(name):
        if p.is_file():
            b = p.read_bytes()
            if sha_bytes(b) == expected_sha:
                return b, str(p)
    for zp in INPUT_ROOT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(zp) as z:
                for member in z.namelist():
                    if Path(member).name == name:
                        b = z.read(member)
                        if sha_bytes(b) == expected_sha:
                            return b, f'{zp}::{member}'
        except zipfile.BadZipFile:
            pass
    raise FileNotFoundError(f'Exact required file not found: {name} / SHA256 {expected_sha}')


def find_components():
    targets = {**BASELINE_CKPT_HASHES, **ADAPTER_HASHES}
    reverse = {h:n for n,h in targets.items()}
    found, sources = {}, {}
    for p in INPUT_ROOT.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in {'.pth','.pt','.ckpt'}:
            continue
        try: h = sha_file(p)
        except Exception: continue
        n = reverse.get(h)
        if n and n not in found:
            found[n] = p; sources[n] = str(p)
    if set(found) != set(targets):
        for zp in INPUT_ROOT.rglob('*.zip'):
            try:
                with zipfile.ZipFile(zp) as z:
                    for m in z.infolist():
                        if not m.filename.lower().endswith(('.pth','.pt','.ckpt')):
                            continue
                        b = z.read(m)
                        h = sha_bytes(b)
                        n = reverse.get(h)
                        if n and n not in found:
                            q = TMP / f'{n}_{h[:12]}.pth'
                            q.write_bytes(b)
                            found[n] = q; sources[n] = f'{zp}::{m.filename}'
            except zipfile.BadZipFile:
                pass
            if set(found) == set(targets): break
    missing = sorted(set(targets)-set(found))
    if missing:
        raise FileNotFoundError('Missing exact locked 08A component(s): ' + ', '.join(missing))
    for n,h in targets.items():
        if sha_file(found[n]) != h:
            raise RuntimeError(f'Component hash mismatch: {n}')
    return ({k:found[k] for k in BASELINE_CKPT_HASHES}, {k:found[k] for k in ADAPTER_HASHES}, sources)


def verify_08a_lock() -> dict:
    b, src = exact_bytes(EXPECTED_08A_LOCK_NAME, EXPECTED_08A_LOCK_SHA)
    lock = json.loads(b)
    if lock.get('lock_id') != LOCK_ID:
        raise RuntimeError('08A lock ID mismatch')
    if float(lock.get('only_inference_change',{}).get('global_residual_gain')) != GLOBAL_RESIDUAL_GAIN:
        raise RuntimeError('08A residual gain mismatch')
    if lock.get('locked_06A_checkpoint_sha256') != {
        'checkpoints/b0_seed_20260820/best.pth': BASELINE_CKPT_HASHES['b0_seed_20260820'],
        'checkpoints/b0_seed_20260821/best.pth': BASELINE_CKPT_HASHES['b0_seed_20260821'],
        'checkpoints/b0_seed_20260822/best.pth': BASELINE_CKPT_HASHES['b0_seed_20260822'],
        'checkpoints/b3_seed_20260820/best.pth': BASELINE_CKPT_HASHES['b3_seed_20260820'],
        'checkpoints/b3_seed_20260821/best.pth': BASELINE_CKPT_HASHES['b3_seed_20260821'],
        'checkpoints/b3_seed_20260822/best.pth': BASELINE_CKPT_HASHES['b3_seed_20260822'],
    }:
        raise RuntimeError('08A 06A checkpoint lock mismatch')
    return {'source':src, 'sha256':EXPECTED_08A_LOCK_SHA, 'lock_id':LOCK_ID, 'global_residual_gain':GLOBAL_RESIDUAL_GAIN}


def load_derm_manifest() -> tuple[pd.DataFrame, str]:
    b, src = exact_bytes(DERM_MANIFEST_NAME, DERM_MANIFEST_SHA)
    df = pd.read_csv(io.BytesIO(b))
    req = {'dataset','image_hint','image_id_norm','group_id','class_name','label','role'}
    if req-set(df.columns): raise RuntimeError('DERM12345 holdout manifest schema mismatch')
    if len(df) != DERM_ROWS: raise RuntimeError(f'DERM12345 row count mismatch: {len(df)}')
    if df.group_id.astype(str).nunique() != DERM_GROUPS: raise RuntimeError('DERM12345 group count mismatch')
    if set(df.dataset.astype(str)) != {'DERM12345'}: raise RuntimeError('DERM12345 dataset field mismatch')
    if set(df.role.astype(str)) != {'domain_holdout'}: raise RuntimeError('DERM12345 role mismatch')
    counts = df.class_name.astype(str).value_counts().to_dict()
    if counts != DERM_CLASS_COUNTS: raise RuntimeError(f'DERM12345 class counts mismatch: {counts}')
    if df[['dataset','image_id_norm']].duplicated().any(): raise RuntimeError('DERM12345 duplicate image IDs')
    return df.reset_index(drop=True), src


def index_derm_images():
    stem, name = defaultdict(list), defaultdict(list)
    count = 0
    for p in INPUT_ROOT.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in EXTS:
            continue
        txt = str(p).lower()
        if not any(t in txt for t in DERM_PATH_TOKENS):
            continue
        stem[p.stem.lower()].append(p); name[p.name.lower()].append(p); count += 1
    if count == 0:
        raise FileNotFoundError('No DERM12345 images found. Attach the DERM12345 image dataset; its Kaggle path/name must contain DERM12345.')
    print('[INDEX] DERM12345 image files:', count)
    return stem, name


def dedup(cands):
    uniq, seen = [], set()
    for p in cands:
        s = str(p)
        if s not in seen:
            seen.add(s); uniq.append(p)
    if len(uniq) <= 1: return uniq
    hs = {sha_file(p) for p in uniq}
    if len(hs) == 1: return [sorted(uniq, key=lambda x:str(x).lower())[0]]
    return uniq


def resolve_derm(df: pd.DataFrame) -> pd.DataFrame:
    stem, name = index_derm_images()
    paths=[]
    for _,r in df.iterrows():
        hint = str(r.image_hint).strip()
        iid = str(r.image_id_norm).strip().lower()
        base = Path(hint.replace('\\','/')).name.lower()
        cands = dedup(name.get(base,[]) + stem.get(iid,[]))
        if len(cands) != 1:
            raise RuntimeError(f'DERM12345 unresolved image {iid}; hint={hint!r}; candidates={cands[:6]}')
        paths.append(str(cands[0]))
    out=df.copy(); out['image_path']=paths
    if out.image_path.duplicated().any(): raise RuntimeError('DERM12345 duplicate resolved paths')
    return out


def validation_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE,IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

class ImageDataset(Dataset):
    def __init__(self, frame): self.frame=frame.reset_index(drop=True); self.tf=validation_transform()
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(r.image_path) as im:
            x=self.tf(ImageOps.exif_transpose(im).convert('RGB'))
        return x, int(r.label), i

def loader(frame,batch):
    return DataLoader(ImageDataset(frame),batch_size=batch,shuffle=False,num_workers=WORKERS,pin_memory=True,persistent_workers=WORKERS>0)


def build_model(family):
    m=models.efficientnet_b0(weights=None) if family=='B0' else models.efficientnet_b3(weights=None)
    m.classifier[1]=nn.Linear(m.classifier[1].in_features,len(CLASSES)); return m

def extract_state(obj):
    if isinstance(obj,dict):
        for k in ('model_state_dict','state_dict','model','ema_state_dict','model_ema_state_dict'):
            v=obj.get(k)
            if isinstance(v,dict) and v and all(torch.is_tensor(x) for x in v.values()): return v
        if obj and all(torch.is_tensor(x) for x in obj.values()): return obj
    raise RuntimeError('No model state_dict found')
def normalize_state_dict(state):
    ks=list(state)
    if ks and all(str(k).startswith('module.') for k in ks): return {str(k)[7:]:v for k,v in state.items()}
    if ks and all(str(k).startswith('_orig_mod.') for k in ks): return {str(k)[10:]:v for k,v in state.items()}
    return state

def load_06a(path,family,device):
    obj=torch.load(path,map_location='cpu',weights_only=False)
    if tuple(obj.get('class_order',())) != CLASSES: raise RuntimeError(f'06A class order mismatch: {path}')
    m=build_model(family); m.load_state_dict(normalize_state_dict(extract_state(obj)),strict=True)
    for p in m.parameters(): p.requires_grad_(False)
    return m.to(device).eval()

@torch.no_grad()
def forward_fp32(m,x,device):
    x=x.to(device=device,dtype=torch.float32,non_blocking=True)
    cm=torch.autocast(device_type='cuda',enabled=False) if device.type=='cuda' else contextlib.nullcontext()
    with cm:
        z=m.features(x); z=m.avgpool(z); z=torch.flatten(z,1); logits=m.classifier(z)
    return z,logits

@torch.no_grad()
def tta7(m,x,device):
    x=x.to(device,non_blocking=True)
    views=(x,torch.flip(x,(3,)),torch.flip(x,(2,)),torch.flip(x,(2,3)),torch.rot90(x,1,(2,3)),torch.rot90(x,2,(2,3)),torch.rot90(x,3,(2,3)))
    total=None
    for v in views:
        with amp(device): q=F.softmax(m(v).float(),dim=1)
        total=q if total is None else total+q
    q=total/len(views)
    bad=~torch.isfinite(q).all(dim=1)
    if bool(bad.any()):
        bx=x[bad].float(); vv=(bx,torch.flip(bx,(3,)),torch.flip(bx,(2,)),torch.flip(bx,(2,3)),torch.rot90(bx,1,(2,3)),torch.rot90(bx,2,(2,3)),torch.rot90(bx,3,(2,3)))
        tot=None
        cm=torch.autocast(device_type='cuda',enabled=False) if device.type=='cuda' else contextlib.nullcontext()
        with cm:
            for v in vv:
                z=F.softmax(m(v).float(),dim=1); tot=z if tot is None else tot+z
        q=q.clone(); q[bad]=tot/len(vv)
    if not torch.isfinite(q).all(): raise RuntimeError('Non-finite TTA7 probability')
    return q

@torch.no_grad()
def extract_family(family,names,paths,frame,device):
    batch=FEATURE_BATCH_B0 if family=='B0' else FEATURE_BATCH_B3
    ms=[load_06a(paths[n],family,device) for n in names]
    dim=ms[0].classifier[1].in_features
    emb=np.zeros((len(frame),dim),np.float32); porig=np.zeros((len(frame),5),np.float32); ptta=np.zeros((len(frame),5),np.float32); filled=np.zeros(len(frame),bool)
    for x,_,idx in tqdm(loader(frame,batch),desc=f'{family} DERM12345 zero-shot'):
        es=ps=ts=None
        for n,m in zip(names,ms):
            e,l=forward_fp32(m,x,device)
            if not torch.isfinite(e).all() or not torch.isfinite(l).all(): raise RuntimeError(f'Non-finite {family}/{n} frozen output')
            p=F.softmax(l,dim=1); t=tta7(m,x,device)
            es=e if es is None else es+e; ps=p if ps is None else ps+p; ts=t if ts is None else ts+t
        ii=idx.numpy(); emb[ii]=(es/len(ms)).cpu().numpy(); porig[ii]=(ps/len(ms)).cpu().numpy(); ptta[ii]=(ts/len(ms)).cpu().numpy(); filled[ii]=True
    if not filled.all(): raise RuntimeError(f'{family} did not fill all DERM12345 rows')
    for a in (emb,porig,ptta):
        if not np.isfinite(a).all(): raise RuntimeError(f'Non-finite {family} DERM12345 array')
    porig/=porig.sum(1,keepdims=True); ptta/=ptta.sum(1,keepdims=True)
    del ms; torch.cuda.empty_cache(); return emb,porig,ptta


def make_stats(p0,p3):
    eps=1e-7; p=.5*p0+.5*p3; p/=p.sum(1,keepdims=True)
    ent=lambda q:-(q*np.log(np.clip(q,eps,1))).sum(1,keepdims=True)
    return np.concatenate([p0,p3,p,ent(p0),ent(p3),ent(p),np.abs(p0-p3),p0.max(1,keepdims=True),p3.max(1,keepdims=True),p.max(1,keepdims=True)],axis=1).astype(np.float32)

class SoftRoutedResidualAdapter(nn.Module):
    def __init__(self,b0_dim,b3_dim,stat_dim):
        super().__init__(); self.b0_proj=nn.Sequential(nn.LayerNorm(b0_dim),nn.Linear(b0_dim,64),nn.GELU()); self.b3_proj=nn.Sequential(nn.LayerNorm(b3_dim),nn.Linear(b3_dim,64),nn.GELU()); self.stat_proj=nn.Sequential(nn.LayerNorm(stat_dim),nn.Linear(stat_dim,32),nn.GELU()); self.trunk=nn.Sequential(nn.Linear(160,128),nn.GELU(),nn.Dropout(DROPOUT)); self.router=nn.Linear(128,len(DOMAINS)); self.experts=nn.ModuleList([nn.Linear(128,len(CLASSES)) for _ in DOMAINS])
    def forward(self,b0,b3,stats):
        h=torch.cat([self.b0_proj(b0),self.b3_proj(b3),self.stat_proj(stats)],1); h=self.trunk(h); r=self.router(h); g=F.softmax(r/ROUTER_TEMPERATURE,1); raw=torch.stack([e(h) for e in self.experts],1); mixed=(g.unsqueeze(-1)*raw).sum(1); return RESIDUAL_LOGIT_CAP*torch.tanh(mixed),r,g

def load_adapters(paths,b0dim,b3dim,sdim,device):
    out=[]; meta={}
    for n in ADAPTER_HASHES:
        obj=torch.load(paths[n],map_location='cpu',weights_only=False)
        if obj.get('experiment')!='PAPER-EXP-06D' or obj.get('selected_common_epoch')!=SELECTED_ADAPTER_EPOCH: raise RuntimeError(f'Adapter metadata mismatch: {n}')
        if tuple(obj.get('class_order',()))!=CLASSES or tuple(obj.get('domain_order',()))!=DOMAINS: raise RuntimeError(f'Adapter order mismatch: {n}')
        st=obj.get('adapter_state_dict'); a=SoftRoutedResidualAdapter(b0dim,b3dim,sdim); a.load_state_dict(st,strict=True)
        if not all(torch.isfinite(v).all() for v in st.values() if torch.is_floating_point(v)): raise RuntimeError(f'Non-finite adapter: {n}')
        for p in a.parameters(): p.requires_grad_(False)
        out.append(a.to(device).eval()); meta[n]={'sha256':sha_file(paths[n]),'seed':int(obj.get('seed'))}
    return out,meta

@torch.no_grad()
def adapter_outputs(adapters,b0,b3,stats,device,batch=1024):
    cs=[]; gs=[]
    for s in range(0,len(b0),batch):
        e=min(s+batch,len(b0)); tb0=torch.from_numpy(b0[s:e]).to(device); tb3=torch.from_numpy(b3[s:e]).to(device); ts=torch.from_numpy(stats[s:e]).to(device); cc=[]; gg=[]
        for a in adapters:
            c,_,g=a(tb0,tb3,ts); cc.append(c); gg.append(g)
        cs.append(torch.stack(cc).mean(0).cpu().numpy()); gs.append(torch.stack(gg).mean(0).cpu().numpy())
    c=np.concatenate(cs).astype(np.float64); g=np.concatenate(gs).astype(np.float64)
    if not np.isfinite(c).all() or not np.isfinite(g).all(): raise RuntimeError('Non-finite adapter ensemble')
    return c,g

def final_probs(base,corr,gain):
    z=np.log(np.clip(base.astype(np.float64),1e-7,1.0))+float(gain)*corr.astype(np.float64); z-=z.max(1,keepdims=True); p=np.exp(z); p/=p.sum(1,keepdims=True); return p


def metrics(y,p):
    pred=p.argmax(1); labels=list(range(5)); recalls=[recall_score(y,pred,labels=[i],average='macro',zero_division=0) for i in labels]; aucs=[]
    for i in labels:
        b=(y==i).astype(int)
        if len(np.unique(b))==2: aucs.append(roc_auc_score(b,p[:,i]))
    return {'n':int(len(y)),'accuracy':float(accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,labels=labels,average='macro',zero_division=0)),'balanced_accuracy':float(np.mean(recalls)),'melanoma_recall':float(recall_score(y,pred,labels=[C2I['MEL']],average='macro',zero_division=0)),'macro_auc':float(np.mean(aucs))}

def per_class(y,p,model):
    pred=p.argmax(1); pr,rc,ff,ss=precision_recall_fscore_support(y,pred,labels=list(range(5)),zero_division=0); rows=[]
    for i,a,b,c,d in zip(range(5),pr,rc,ff,ss): rows.append({'model':model,'class':CLASSES[i],'support':int(d),'precision':float(a),'recall':float(b),'f1':float(c)})
    return pd.DataFrame(rows)

def save_predictions(path,df,p,gates,model):
    o=df[['dataset','image_id_norm','group_id','class_name','label','role']].copy(); pred=p.argmax(1); o['model_name']=model; o['predicted_label']=pred; o['predicted_class']=[CLASSES[i] for i in pred]; o['correct']=(pred==o.label.to_numpy(int)).astype(int)
    for i,c in enumerate(CLASSES): o[f'prob_{c}']=p[:,i]
    for j,d in enumerate(DOMAINS): o[f'gate_{d}']=gates[:,j]
    o['router_argmax_expert']=[DOMAINS[i] for i in gates.argmax(1)]; o.to_csv(path,index=False)

def material_index():
    rows=[]
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.name!='paper_material_index.csv' and p.suffix.lower()!='.zip': rows.append({'relative_path':str(p.relative_to(OUT)),'bytes':p.stat().st_size,'sha256':sha_file(p)})
    df=pd.DataFrame(rows); df.to_csv(OUT/'paper_material_index.csv',index=False); return df

def make_archives():
    rz=WORKING_ROOT/'PAPEREXP08B_Locked_08A_DERM12345_ZeroShot_Results.zip'; cz=WORKING_ROOT/'PAPEREXP08B_Locked_08A_DERM12345_ZeroShot_Recovery.zip'; files=[p for p in OUT.rglob('*') if p.is_file()]
    for t in (rz,cz):
        with zipfile.ZipFile(t,'w',compression=zipfile.ZIP_DEFLATED) as z:
            for p in files: z.write(p,arcname=str(p.relative_to(OUT)))
    return rz,cz


def main():
    if not torch.cuda.is_available(): raise RuntimeError('PAPER-EXP-08B requires a CUDA GPU for exact TTA7 inference.')
    device=torch.device('cuda')
    print('[GPU]',torch.cuda.get_device_name(0)); print('[CODE VERSION]',CODE_VERSION); print('[LOCK]',LOCK_ID); print('[STATUS] ONCE-ONLY DERM12345 ZERO-SHOT; NO TRAINING / NO ADAPTATION / NO SELECTION')

    lock_audit=verify_08a_lock(); base_paths,adapter_paths,sources=find_components()
    preflight={'experiment':EXPERIMENT,'code_version':CODE_VERSION,'locked_model':'PAPER-EXP-08A','lock_id':LOCK_ID,'global_residual_gain':GLOBAL_RESIDUAL_GAIN,'08A_lock':lock_audit,'component_sources':sources,'DERM12345_manifest_read':False,'DERM12345_predictions_seen':False,'training_performed':False,'adaptation_performed':False,'calibration_performed':False,'threshold_search':False,'fusion_search':False,'model_selection':False}
    json_dump(preflight,AUDIT/'preflight_before_DERM12345_open.json')

    print('[IRREVERSIBLE STEP] 08A lock verified. Opening the pre-frozen DERM12345 holdout manifest once.')
    derm,manifest_src=load_derm_manifest(); preflight['DERM12345_manifest_read']=True; preflight['DERM12345_manifest_source']=manifest_src; json_dump(preflight,AUDIT/'preflight_before_DERM12345_open.json')
    derm=resolve_derm(derm)

    b0e,b0o,b0t=extract_family('B0',B0_NAMES,base_paths,derm,device); b3e,b3o,b3t=extract_family('B3',B3_NAMES,base_paths,derm,device)
    base=.5*b0t.astype(np.float64)+.5*b3t.astype(np.float64); base/=base.sum(1,keepdims=True); stats=make_stats(b0o,b3o); adapters,adapter_meta=load_adapters(adapter_paths,b0e.shape[1],b3e.shape[1],stats.shape[1],device); corr,gates=adapter_outputs(adapters,b0e,b3e,stats,device)
    p06d=final_probs(base,corr,1.0); p08a=final_probs(base,corr,GLOBAL_RESIDUAL_GAIN); y=derm.label.to_numpy(int)
    m06a=metrics(y,base); m06d=metrics(y,p06d); m08a=metrics(y,p08a)

    save_predictions(PRED/'PAPEREXP08A_DERM12345_zero_shot_predictions.csv',derm,p08a,gates,'PAPER-EXP-08A locked')
    pd.DataFrame([{'model':'PAPER-EXP-06A parent',**m06a},{'model':'PAPER-EXP-06D gain=1.00 descriptive',**m06d},{'model':'PAPER-EXP-08A final gain=1.05',**m08a}]).to_csv(TABLES/'Table1_DERM12345_zero_shot_metrics.csv',index=False)
    pd.concat([per_class(y,base,'PAPER-EXP-06A parent'),per_class(y,p06d,'PAPER-EXP-06D gain=1.00 descriptive'),per_class(y,p08a,'PAPER-EXP-08A final gain=1.05')],ignore_index=True).to_csv(TABLES/'Table2_DERM12345_per_class_metrics.csv',index=False)
    cm=confusion_matrix(y,p08a.argmax(1),labels=list(range(5))); pd.DataFrame(cm,index=[f'true_{c}' for c in CLASSES],columns=[f'pred_{c}' for c in CLASSES]).to_csv(TABLES/'Table3_08A_DERM12345_confusion_matrix.csv')
    router={'n':len(derm),'router_argmax_distribution':{DOMAINS[i]:int(np.sum(gates.argmax(1)==i)) for i in range(len(DOMAINS))},'mean_gate_weights':{DOMAINS[i]:float(gates[:,i].mean()) for i in range(len(DOMAINS))}}
    json_dump(router,TABLES/'Table4_DERM12345_router_behavior.json')

    summary={'experiment':EXPERIMENT,'title':TITLE,'code_version':CODE_VERSION,'status':'COMPLETE_ONCE_ONLY_ZERO_SHOT_DERM12345','locked_model':'PAPER-EXP-08A','lock_id':LOCK_ID,'global_residual_gain':GLOBAL_RESIDUAL_GAIN,'DERM12345_manifest_sha256':DERM_MANIFEST_SHA,'DERM12345_rows':DERM_ROWS,'DERM12345_groups':DERM_GROUPS,'PAPER_EXP_08A_zero_shot_metrics':m08a,'PAPER_EXP_06D_gain1_descriptive_metrics':m06d,'PAPER_EXP_06A_parent_descriptive_metrics':m06a,'adapter_metadata':adapter_meta,'training_performed':False,'target_domain_adaptation':False,'calibration':False,'threshold_search':False,'fusion_weight_search':False,'TTA_search':False,'model_selection_after_DERM12345':False,'post_evaluation_model_change_permitted':False,'decision':'REPORT_LOCKED_08A_ZERO_SHOT_RESULT_REGARDLESS_OF_PERFORMANCE'}
    json_dump(summary,OUT/'PAPEREXP08B_DERM12345_ZEROSHOT_SUMMARY.json')
    protocol={'experiment':EXPERIMENT,'scientific_role':'Independent once-only zero-shot evaluation of the already locked PAPER-EXP-08A generalized configuration on DERM12345.','locked_before_DERM12345_outcomes':True,'model':'PAPER-EXP-08A exact lock','gain':GLOBAL_RESIDUAL_GAIN,'manifest_sha256':DERM_MANIFEST_SHA,'training':'none','adaptation':'none','selection':'none','calibration':'none','thresholding':'argmax','hard_rule':'DERM12345 outcome cannot change PAPER-EXP-08A or trigger post-hoc selection.'}
    json_dump(protocol,OUT/'paper_exp08b_protocol.json')
    source_req={'required_final_model_recovery_sha256_if_zip_preserved':EXPECTED_08A_RECOVERY_SHA,'required_08A_lock_sha256':EXPECTED_08A_LOCK_SHA,'required_06A_checkpoint_sha256':BASELINE_CKPT_HASHES,'required_06D_adapter_sha256':ADAPTER_HASHES,'required_DERM12345_manifest_sha256':DERM_MANIFEST_SHA,'note':'Recovery of numerical inference requires the exact PAPER-EXP-08A Final Locked Generalized Model Recovery plus the DERM12345 image dataset.'}
    json_dump(source_req,OUT/'PAPEREXP08B_SOURCE_REQUIREMENTS.json')
    final_dec={'experiment':EXPERIMENT,'locked_model':'PAPER-EXP-08A','DERM12345_opened_once':True,'DERM12345_used_for_training_or_selection':False,'decision':'KEEP_PAPER_EXP_08A_LOCKED_AND_REPORT_ZERO_SHOT_RESULT','future_supervised_DERM12345_adaptation_if_any':'must be separately labeled and cannot replace this zero-shot result'}
    json_dump(final_dec,OUT/'PAPEREXP08B_FINAL_DECISION.json')
    material_index(); rz,cz=make_archives()
    print('\n[FINAL PAPER-EXP-08A DERM12345 ZERO-SHOT METRICS]'); print(json.dumps(m08a,indent=2)); print('[RESULTS ZIP]',rz); print('[RESULTS SHA256]',sha_file(rz)); print('[RECOVERY ZIP]',cz); print('[RECOVERY SHA256]',sha_file(cz)); print('[DONE] PAPER-EXP-08B complete; PAPER-EXP-08A remains locked regardless of outcome.')

if __name__=='__main__': main()


[GPU] Tesla T4
[CODE VERSION] PAPEREXP08B_LOCKED_08A_DERM12345_ZEROSHOT_2026-08-30_v1
[LOCK] PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK_2026-08-30_v1
[STATUS] ONCE-ONLY DERM12345 ZERO-SHOT; NO TRAINING / NO ADAPTATION / NO SELECTION
[IRREVERSIBLE STEP] 08A lock verified. Opening the pre-frozen DERM12345 holdout manifest once.
[INDEX] DERM12345 image files: 12345


B0 DERM12345 zero-shot:   0%|          | 0/73 [00:00<?, ?it/s]

B3 DERM12345 zero-shot:   0%|          | 0/146 [00:00<?, ?it/s]


[FINAL PAPER-EXP-08A DERM12345 ZERO-SHOT METRICS]
{
  "n": 2321,
  "accuracy": 0.925463162429987,
  "macro_f1": 0.6984605969129508,
  "balanced_accuracy": 0.748690528329432,
  "melanoma_recall": 0.7682926829268293,
  "macro_auc": 0.9682713537767309
}
[RESULTS ZIP] /kaggle/working/PAPEREXP08B_Locked_08A_DERM12345_ZeroShot_Results.zip
[RESULTS SHA256] ad24b282d75451a0aeffe8099a8ca7f75de716ff31a9b0a2a18ff151c28a37f3
[RECOVERY ZIP] /kaggle/working/PAPEREXP08B_Locked_08A_DERM12345_ZeroShot_Recovery.zip
[RECOVERY SHA256] ad24b282d75451a0aeffe8099a8ca7f75de716ff31a9b0a2a18ff151c28a37f3
[DONE] PAPER-EXP-08B complete; PAPER-EXP-08A remains locked regardless of outcome.
